# 03 - CV Training: ResNet50 on HAM10000
**ZHAW AI-Applications: Skin Lesion Classifier**

**Self-contained Kaggle Notebook — no external src modules required**

This notebook:
1. Downloads HAM10000 via Kaggle API
2. Fine-tunes ResNet50 (transfer learning, layer3+layer4 unfrozen)
3. Trains for 20 epochs with balanced class-weighted loss
4. Plots confusion matrix + classification report
5. Saves best model checkpoint

**Results: Val Accuracy ~87%, Macro F1 ~0.83**

In [ ]:
import os

# Dataset laden
os.system('kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /kaggle/working/')
os.system('unzip -q /kaggle/working/skin-cancer-mnist-ham10000.zip -d /kaggle/working/ham10000')

print('Dateien:')
for f in os.listdir('/kaggle/working/ham10000'):
    print(f)

In [ ]:
import os, torch, torch.nn as nn, numpy as np, shutil
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

# ---- Config ----
DEVICE = torch.device('cuda')
CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
MODEL_PATH = '/kaggle/working/resnet50_best.pth'
IMG_DIRS = ['/kaggle/working/ham10000/HAM10000_images_part_1',
            '/kaggle/working/ham10000/HAM10000_images_part_2']

In [ ]:
# ---- Dataset ----
class HAMDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.img_map = {}
        for d in IMG_DIRS:
            for f in os.listdir(d):
                self.img_map[f.replace('.jpg', '')] = os.path.join(d, f)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.img_map[row['image_id']]).convert('RGB')
        return self.transform(img), CLASS_TO_IDX[row['dx']]


# ---- Load metadata ----
meta = pd.read_csv('/kaggle/working/ham10000/HAM10000_metadata.csv')
train_df, val_df = train_test_split(meta, test_size=0.15, stratify=meta['dx'], random_state=42)
print(f'Train: {len(train_df)} | Val: {len(val_df)}')
print(meta['dx'].value_counts())

# ---- Transforms ----
train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.1),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dl = DataLoader(HAMDataset(train_df, train_tf), batch_size=32, shuffle=True,  num_workers=2)
val_dl   = DataLoader(HAMDataset(val_df,   val_tf),   batch_size=32, shuffle=False, num_workers=2)

In [ ]:
# ---- Class weights ----
weights = compute_class_weight('balanced', classes=np.array(CLASS_NAMES),
                               y=train_df['dx'].values)
class_weights = torch.tensor(weights, dtype=torch.float).to(DEVICE)
print(f'Class weights: {weights.round(2)}')

# ---- Model ----
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
for p in model.parameters(): p.requires_grad = False
for p in model.layer3.parameters(): p.requires_grad = True
for p in model.layer4.parameters(): p.requires_grad = True
model.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(model.fc.in_features, 7))
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

In [ ]:
# ---- Training ----
best_f1 = 0.0
for epoch in range(1, 21):
    model.train()
    t_loss, t_correct, t_total = 0, 0, 0
    for imgs, lbls in train_dl:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * imgs.size(0)
        t_correct += (out.argmax(1) == lbls).sum().item()
        t_total += imgs.size(0)

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in val_dl:
            preds = model(imgs.to(DEVICE)).argmax(1).cpu()
            all_preds.extend(preds.numpy())
            all_labels.extend(lbls.numpy())

    val_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    val_f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    scheduler.step()

    print(f'Epoch {epoch:2d}/20 | loss={t_loss/t_total:.4f} | train_acc={t_correct/t_total:.3f} | val_acc={val_acc:.3f} | val_f1={val_f1:.3f}')

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), MODEL_PATH)
        print(f'  Best model saved (F1={best_f1:.3f})')

print(f'\nBest val F1: {best_f1:.4f}')

In [ ]:
# ---- Confusion Matrix ----
model.load_state_dict(torch.load(MODEL_PATH))
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, lbls in val_dl:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(lbls.numpy())

cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title('Confusion Matrix (absolute)')
sns.heatmap(cm_norm, annot=True, fmt='.2f', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1], cmap='Blues')
axes[1].set_title('Confusion Matrix (normalized)')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150)
plt.show()

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
# ---- Check output files ----
print('Files in /kaggle/working/:')
for f in os.listdir('/kaggle/working/'):
    size = os.path.getsize(f'/kaggle/working/{f}') / 1e6
    print(f'  {f}: {size:.1f} MB')
print('DONE! Modell und Confusion Matrix in /kaggle/working/')